# **Chess Win Predictor**

## **Abstract**
This project studies how to predict the winner of a chess game using information that reflects how humans actually play under practical conditions. Instead of relying only on engine evaluation, the project focuses on player Elo, remaining clock time, move-by-move time usage, and board state extracted from Lichess PGN files. The first preprocessing step filters the raw Lichess monthly export to keep only completed rapid games played in the regular 10-minute format (`600+0`). This creates a cleaner and more consistent dataset for later feature extraction and modeling. The current pipeline compares linear, tree-based, feed-forward neural, and recurrent neural models while explicitly avoiding future-information leakage. In addition to standard held-out testing, the project now uses landmark evaluation every 5 full moves so that early-game and late-game prediction quality can be assessed separately.


## **Introduction**

The goal of this project is to estimate chess win probability using features that matter in human play, such as player rating, time pressure, and move-by-move game context. Traditional engine evaluations describe who should be better under perfect play, but real games are strongly affected by clock management, practical mistakes, and rating differences. For that reason, a human-centered win predictor can be more realistic than a pure engine-based score.

The dataset comes from the Lichess Open Database, which provides large PGN exports with game metadata and clock annotations inside move comments. A central challenge is data leakage: the model must only use information available at the moment of prediction and must not accidentally learn from future moves, end-of-game tags, or derived signals that reveal the final outcome. Another challenge is data quality, because the raw export contains many time controls and termination types that are not equally suitable for this project.

To make the dataset more consistent, the pipeline begins by filtering the raw PGN file to keep only completed rapid games in the standard 10-minute format. Later stages will convert these filtered games into structured examples for training and evaluation.

In terms of originality, the project is not simply trying to predict the best chess move or reproduce an engine evaluation. Instead, it focuses on a human-centered question: given a real over-the-board style online rapid game, how well can the final outcome be predicted from practical factors such as rating, time pressure, and a compact board summary? The combination of board-aware but non-engine features, landmark-based probability evaluation, and recurrent modeling of game prefixes differentiates the project from standard engine-centric chess prediction tasks and from generic solved competition templates.


## **Methodology**
The first step of the methodology is dataset filtering. The raw monthly Lichess PGN export is extremely large and contains many different game speeds and termination types, so a dedicated preprocessing script named `filter_lichess_pgn.py` was created to stream through the file one game at a time instead of loading it into memory.

This script reads each PGN game as a block, extracts the headers, and keeps only games that satisfy two conditions:
1. `TimeControl == "600+0"`, which corresponds to a regular 10-minute rapid game with no increment.
2. `Termination == "Normal"`, which is used here as the definition of a completed game.

The filtering stage is important because it removes games from other formats such as bullet or blitz and reduces noise from terminations that are less useful for the current study. It also keeps the preprocessing reproducible, since the exact inclusion rules are encoded in a standalone script rather than being applied manually.

The second preprocessing step uses `extract_lichess_features.py`, which converts the filtered PGN into move level tabular rows. Each row is a snapshot immediately after a move has been played and includes only information available at that moment. The current baseline extractor focuses on human centered features that are already present in the PGN data: player Elo, side to move, remaining clock for both players, clock ratios, clock differences, and the amount of time spent on the latest move.

In addition, the extractor derives lightweight move shape features directly from SAN notation. These include whether the move is a capture, check, checkmate, castle, or promotion, as well as the SAN string length. These features are simple, cheap to compute, and still avoid the need for engine analysis or future information.

A richer extractor named `extract_lichess_board_features.py` was then added using `python-chess`. This version reconstructs the board after each move and computes board derived features such as material counts for each side, material difference, legal move count, halfmove clock, castling rights, bishop pair indicators, and insufficient material status. These features are still legal from a data leakage perspective because they are computed only from the current board state and the current move history.

In this report, `board-aware` means that the model is not only reading metadata like Elo and clock times, but is also given a compact summary of the actual chess position on the board at that moment. For example, instead of only knowing that White spent a long time on the last move, the model can also know whether White is up material, whether castling rights still exist, how many legal moves are available, or whether the position is close to a trivial draw. This is still much simpler than feeding the model a full engine evaluation, but it gives the model real chess state context.

The role of `python-chess` is to make that possible safely and correctly. PGN stores games as move text, not as full board snapshots after every move. `python-chess` starts from the standard initial position, replays the moves one by one, and updates the internal board after each move. Once the board has been reconstructed, the script can query it for things like piece counts, legal move count, castling rights, or whether the side to move is in check. In other words, `python-chess` is the rules engine that turns PGN text into a usable board state for feature extraction.

During model training, the feature set intentionally excludes `fullmove_number` and any total-game-length information as direct inference features. The reason is that the final system should not rely on explicit future information or move count information if that quantity is not intended to be provided at prediction time. This keeps the train/test setup consistent and matches the proposal's goal of preventing the model from cheating by seeing future context.

To evaluate the models in a way that better reflects the proposal's probability prediction goal, the project now uses a landmarking approach. Instead of only averaging over all move snapshots, the evaluation script `evaluate_landmarks.py` tests each model every 5 full moves. For landmark `k`, it uses the position after Black's `k`th move, which corresponds to ply index `2k`. This gives one snapshot per game at each landmark and avoids overweighting longer games.

A sequence model was also added to better match the temporal nature of chess. The script `train_rnn_landmarks.py` builds one training example for each game prefix ending at a 5 move landmark and trains a basic Keras `SimpleRNN` on those prefix sequences. This allows the model to use the recent move history directly instead of relying only on snapshot features.

For scalability, a parallel CPU version of the board-aware extractor was also tested. On a `1,000` game benchmark, the serial extractor took about `56.4` seconds while a `4` worker parallel version reduced that to about `19.8` seconds, a speedup of roughly `2.85x`. Later in the project, a sharded sequence pipeline was added with `extract_rnn_game_shards_parallel.py`, which saves full board-aware game sequences directly to NPZ shards instead of first materializing a giant move level CSV. This made it possible to train recurrent models from reusable sequence shards and reduced repeated parsing work across larger RNN experiments.

TensorFlow in the local Windows environment was CPU only, but a CUDA enabled PyTorch setup was later installed and verified on the local RTX `3060`. A new trainer, `train_simplernn_from_shards_torch.py`, was added to read the NPZ shards with persistent workers, pinned memory, mixed precision, periodic checkpoints, and resumable training. This made the later full-dataset SimpleRNN experiment feasible while preserving the same 5 move landmark evaluation design.

This design gives a strong first baseline without requiring engine evaluations or future information. It also makes it possible to train and test models quickly on development subsets before adding richer board-state features later.


## **Experimental Setup**
The raw dataset is the February 2026 Lichess standard rated PGN export. Before modeling, the file is filtered with `filter_lichess_pgn.py` to keep only `600+0` games with `Termination = "Normal"`. This produced a focused rapid-only dataset that is much better aligned with the project objective than the original mixed-format export.

The script is run as follows:

```python
!python scripts/filter_lichess_pgn.py \
    --input lichess_db_standard_rated_2026-02/lichess_db_standard_rated_2026-02.pgn \
    --output data/lichess_rapid_10_0_completed.pgn \
    --time-control 600+0 \
    --termination Normal
```

On the full file, the script processed `84,600,043` games and kept `8,633,881` games. This filtered output is the starting point for downstream feature extraction, train/validation/test splitting, and model comparison.

The next preprocessing stage extracts move-level features:

```python
!python scripts/extract_lichess_features.py \
    --input data/lichess_rapid_10_0_completed.pgn \
    --output data/dev_move_features_1000_games.csv \
    --max-games 1000
```

A richer board-aware version can also be extracted with:

```python
!python scripts/extract_lichess_board_features.py \
    --input data/lichess_rapid_10_0_completed.pgn \
    --output data/dev_board_features_1000_games.csv \
    --max-games 1000
```

For a first development experiment, `1,000` filtered games were converted into `65,188` move-level rows. Models were then trained on these tabular features using a game-level `train/validation/test` split to prevent rows from the same game appearing in multiple subsets.

To test whether the neural network approach benefits from more data, the same board-aware extraction pipeline was also scaled to `10,000` games, producing `639,885` move-level rows. This larger subset was used for a deeper MLP experiment while keeping the same game-level split strategy.
For the largest recurrent experiment, the preprocessing pipeline switched from flat CSV export to sharded sequence storage. The script `extract_rnn_game_shards_parallel.py` streamed the filtered PGN directly, reconstructed each board-aware game sequence, and saved the result into reusable NPZ shards split by game into train/validation/test. On the final full extraction over all `8,633,881` filtered rapid games, this produced `6,214,945` training games, `690,252` validation games, and `1,726,823` test games across `1243/139/346` train/validation/test shards respectively.

The model families and hyperparameters used so far are the following:
- Light logistic regression: `StandardScaler` on numeric features, `OneHotEncoder` on categorical features, `LogisticRegression(max_iter=400, random_state=42)`.
- Light histogram gradient boosting: passthrough numeric features, dense one-hot categorical features, `HistGradientBoostingClassifier(max_depth=8, learning_rate=0.1, max_iter=200, min_samples_leaf=40, random_state=42)`.
- Board logistic regression: same logistic-regression pipeline as above, but trained on the board-aware feature set.
- Board calibrated SVM: `StandardScaler` on numeric features, sparse `OneHotEncoder` on categorical features, then `CalibratedClassifierCV(LinearSVC(C=1.5, class_weight="balanced", dual="auto", max_iter=6000, random_state=42), method="sigmoid", cv=3)`. This was used instead of a full kernel SVC so the model could still train on the `10,000`-game board-aware dataset while producing calibrated probabilities for log loss and landmark evaluation.
- Board histogram gradient boosting: same gradient-boosting pipeline as above, but trained on the board-aware feature set.
- Board XGBoost: passthrough numeric features, dense one-hot categorical features, then `XGBClassifier(objective="multi:softprob", num_class=3, n_estimators=300, max_depth=8, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, min_child_weight=2, tree_method="hist", eval_metric="mlogloss", random_state=42)`.
- Board MLP: `MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", solver="adam", alpha=1e-4, batch_size=256, learning_rate_init=1e-3, max_iter=100, early_stopping=True, validation_fraction=0.1, n_iter_no_change=10, random_state=42)`.
- Board deep MLP: `MLPClassifier(hidden_layer_sizes=(256, 128, 64), activation="relu", solver="adam", alpha=5e-5, batch_size=512, learning_rate_init=7e-4, max_iter=150, early_stopping=True, validation_fraction=0.1, n_iter_no_change=12, random_state=42)`.
- Board wide MLP: `MLPClassifier(hidden_layer_sizes=(512, 256, 128), activation="relu", solver="adam", alpha=1e-4, batch_size=512, learning_rate_init=5e-4, max_iter=180, early_stopping=True, validation_fraction=0.1, n_iter_no_change=15, random_state=42)`.
- Board balanced MLP: `MLPClassifier(hidden_layer_sizes=(256, 128, 64), activation="relu", solver="adam", alpha=3e-4, batch_size=512, learning_rate_init=4e-4, max_iter=200, early_stopping=True, validation_fraction=0.1, n_iter_no_change=18, random_state=42)` plus training-only draw oversampling factor `3.0`.
- Basic RNN landmark model: Keras `SimpleRNN(64)` followed by `Dense(32, relu)` and a `Dense(3, softmax)` output layer, trained with `Adam(learning_rate=1e-3)`, `batch_size=64`, `max_epochs=20`, and early stopping on validation loss. The input is a padded prefix sequence of board-aware move features ending at each 5-move landmark.
- Shard-based PyTorch SimpleRNN: `nn.RNN(input_size=41, hidden_size=64)` followed by `Linear(64,32)`, `ReLU`, and `Linear(32,3)`, trained with `Adam(learning_rate=1e-3)`, `batch_size=256`, `max_epochs=12`, CUDA mixed precision, `num_workers=6`, `pin_memory=True`, `persistent_workers=True`, epoch checkpoints, and the same 5-move landmark prefixes generated from NPZ game shards.
- Additional recurrent variants tested on `1,000` games: `GRU(64)`, `LSTM(64)`, stacked `GRU(64,64)`, `GRU(64)` with dropout `0.2` and recurrent dropout `0.1`, `GRU(64)` with `learning_rate=5e-4`, `batch_size=128`, `max_epochs=40`, `GRU(64)` with landmark weighting between moves `20` and `50` using factor `1.5`, `GRU(64)` with draw-weight factor `2.0`, and a combined variant using both dropout regularization and midgame landmark weighting.


## **How to Use This Notebook in Google Colab**

This notebook is the main submission artifact and is designed to be fully runnable in Google Colab. There are two levels of reproducibility:

### Level 1 — Lightweight Colab run (recommended for graders)

This is the fast path. It clones the repository, loads the archived results from `artifacts/reported_results/`, and optionally runs a small end-to-end smoke test on a 60-game sample dataset. **No large files required.**

**Steps:**

1. **Run Cell 6 (Setup)** — clones the repository and installs dependencies automatically.
2. **Run Cell 7 (Load results)** — prints the SimpleRNN scaling table and the 250k PyTorch RNN reported metrics directly from the tracked artifact files.
3. **Run Cell 8 (Smoke test, optional)** — runs the full pipeline (filter → extract → shard → train) on a tiny 60-game sample in a few minutes on a standard Colab CPU runtime.

All cells can be run top-to-bottom with **Runtime → Run all**.

---

### Level 2 — Full reproduction (requires large data files)

> **Important context:** The main experimental work for this project was done **outside of Google Colab**, on a local Windows machine with an NVIDIA RTX 3060 GPU. The reasons are practical: the raw Lichess PGN export for February 2026 is **27.7 GB compressed** and contains **84.6 million games**. After filtering to completed 10-minute rapid games, the result is still **8.6 million games**. Preprocessing and training at that scale is not feasible on a free Colab runtime.

The full reproduction path (documented in detail in `COLAB_REPRODUCIBILITY.md`) requires either:
- Uploading the pre-filtered `lichess_rapid_10_0_completed.pgn` to Colab or Google Drive, **or**
- Starting from the raw Lichess PGN (even larger)

and then running the extraction and training scripts documented in the Experimental Setup section. The archived results in `artifacts/reported_results/` were generated from those full runs and represent the numbers reported in the tables below.

---

The notebook therefore serves as the **readable report layer** — the external scripts in `scripts/` contain the modular, reusable pipeline code. Links to each key script are provided inline in the Methodology section.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/thermoNuke1/ml-PROJECT.git"
REPO_NAME = "ml-PROJECT"
REPO_PATH = Path(f"/content/{REPO_NAME}")

os.chdir("/content")

if not REPO_PATH.is_dir():
    !git clone {REPO_URL} {REPO_PATH}
else:
    print("Already cloned - pulling latest changes")
    !git -C {REPO_PATH} pull

requirements_path = REPO_PATH / "requirements.txt"
if requirements_path.exists():
    !pip install -q -r {requirements_path}
else:
    print("No requirements.txt found - using the current notebook environment")

repo_path_str = str(REPO_PATH)
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)

os.chdir(REPO_PATH)
print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {repo_path_str}")


In [1]:
from pathlib import Path
import json
import pandas as pd

# Archived results are tracked in artifacts/reported_results/
# (the data/ directory is git-ignored and not present in Colab)
artifacts_dir = Path('artifacts/reported_results')

scaling_path = artifacts_dir / 'simplernn_scaling_summary.csv'
if scaling_path.exists():
    scaling_df = pd.read_csv(scaling_path)
    print("SimpleRNN scaling summary:")
    display(scaling_df)
else:
    print(f'scaling summary not found at {scaling_path}')

rnn_path = artifacts_dir / 'torch_rnn_landmark_board_250000_with_history.json'
if rnn_path.exists():
    payload = json.loads(rnn_path.read_text(encoding='utf-8'))
    print("
250k PyTorch SimpleRNN reported results:")
    print({k: payload[k] for k in ['test_accuracy', 'test_log_loss', 'train_games', 'test_games']})
else:
    print(f'250k RNN metrics not found at {rnn_path}')


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Optional smoke test: runs the full pipeline on a 60-game sample dataset.
# Takes ~2-4 minutes on a standard Colab CPU runtime.
# Outputs go to artifacts/colab_runs/ (board features CSV, logreg metrics, RNN metrics).

import subprocess, sys

result = subprocess.run(
    [sys.executable, 'scripts/run_colab_repro.py', '--run-rnn-smoke', '--rnn-epochs', '2'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)


## **Experimental Results**
The first baseline experiment was run on the `1,000`-game development subset created from the filtered rapid dataset. The model used only tabular metadata and clock features extracted directly from the PGN, with no engine evaluation and no full board-state encoding yet. This makes the result a useful reference point for measuring later improvements.

The baseline setup used:
- `720` training games, `80` validation games, and `200` test games
- `46,436` training rows, `5,202` validation rows, and `13,550` test rows
- a game-level split to avoid leakage across move snapshots from the same game
- logistic regression on Elo, clock, and SAN-derived move-shape features

Using the lighter feature set based mainly on Elo, clocks, and SAN-shape features, the logistic-regression baseline achieved a validation accuracy of about `0.477` with validation log loss `0.893`, and a test accuracy of about `0.457` with test log loss `0.911`. The model performed better on white-win and black-win classes than on draws, which is expected because draws were much less frequent in this development subset.

A more complex tree-based model was then tested using histogram gradient boosting, which is a stronger non-linear alternative to logistic regression and is conceptually closer to the boosted-tree direction proposed for the project. On the same light-feature split, this model achieved a test accuracy of about `0.461`, which is only slightly better than logistic regression. However, its test log loss increased sharply to about `2.221`, and it essentially failed to model the draw class. This suggested that the more complex model was not the main missing ingredient.

After adding board-derived features with `python-chess`, the results improved much more clearly. With the richer board-aware feature set, logistic regression reached a validation accuracy of about `0.604` and a test accuracy of about `0.572`, with test log loss reduced to about `0.837`. These results were obtained while explicitly excluding `fullmove_number` from the final inference feature set. On the same board-aware dataset, histogram gradient boosting achieved a test accuracy of about `0.539` with test log loss about `1.826`, which remained worse than logistic regression in both calibration and overall accuracy.

A direct boosted-tree experiment was also run on the `10,000` game board-aware dataset. This XGBoost model reached about `0.598` test accuracy with test log loss about `0.844`. That is clearly better than the earlier histogram gradient boosting baseline and much more stable than the MLP family, but it still did not beat the `10,000`-game board logistic-regression model. In particular, XGBoost continued to struggle on draws, with draw recall remaining very low, so its main value in this project is as a strong additional tree-based comparison rather than as the best final model.

A calibrated linear SVM was also tested on the board-aware features, using feature scaling, class balancing, and sigmoid probability calibration. On `1,000` games, this model reached about `0.578` test accuracy with test log loss about `0.818`, which made it competitive with board logistic regression but still slightly worse in both overall test accuracy and landmark calibration. When scaled to `10,000` games, the SVM improved further to about `0.610` test accuracy with test log loss about `0.795`, nearly matching the `10,000`-game board logistic model. Its average landmark accuracy reached about `0.657`, with accuracies around `0.527` at move `5`, `0.642` at move `20`, and `0.679` at move `40`. This makes the calibrated linear SVM a strong additional tabular baseline, even though it still did not clearly surpass the board logistic-regression model.

To reflect the neural-network direction proposed in the project plan, a multilayer perceptron (MLP) classifier was also trained on the board-aware features. A smaller MLP achieved a test accuracy of about `0.541`, which is below the board-aware logistic regression result, and a test log loss of about `3.280`, which indicates poorly calibrated probability estimates. However, unlike the tree model, the MLP was able to recover at least some of the draw class, reaching roughly `0.20` precision and recall on draws in the test set.

A deeper neural network was then tested on both `1,000` and `10,000` games. On the `1,000`-game subset, the deeper MLP reached about `0.543` test accuracy with test log loss about `3.668`. When scaled to `10,000` games, the same model improved slightly to about `0.545` test accuracy and test log loss about `3.072`. This suggests that the neural network does benefit from additional data, especially for stabilizing the draw class, but it still does not match the overall accuracy or probability calibration of the board-aware logistic-regression model.

The board-aware logistic-regression model was also scaled from `1,000` to `10,000` games. On the larger dataset it reached about `0.611` test accuracy with test log loss about `0.795`, and its average landmark accuracy increased to about `0.665` with average landmark Brier score about `0.455`. The move-level landmark profile also became stronger: about `0.525` accuracy at move `5`, `0.559` at move `10`, `0.639` at move `20`, `0.694` at move `30`, and `0.683` at move `40`. This makes the `10,000`-game board logistic-regression model one of the strongest calibrated snapshot baselines in the project.

Because chess is naturally sequential, a basic recurrent neural network was also tested on the board-aware landmark data. Instead of using only one snapshot at a time, this model was trained on game prefixes ending at each 5-move landmark. On the `1,000`-game board-aware subset, the resulting basic `SimpleRNN` achieved about `0.598` test accuracy with test log loss about `0.807` over the landmark-sequence test set. Its average landmark accuracy was about `0.628`, and its average landmark Brier score was about `0.471`, which is better than the board-aware logistic-regression landmark average. At move `20`, landmark accuracy was about `0.623`, and by move `40` it reached about `0.783`, suggesting that the recurrent model benefits from explicitly modeling temporal context.

The same RNN architecture was then scaled to the `10,000`-game board-aware subset. On this larger dataset, the sequence model improved further to about `0.615` test accuracy with test log loss about `0.774`. The landmark profile remained strong: about `0.529` accuracy at move `5`, `0.627` at move `20`, and `0.701` at move `40`, with Brier score improving steadily through the game. This is the strongest sequence-based result obtained so far and provides evidence that recurrent models may benefit more directly from additional data than the feed-forward MLP variants.

A focused recurrent-model tuning sweep was also run on the `1,000`-game board-aware subset. The strongest overall result came from a combined `GRU(64)` variant using dropout `0.2`, recurrent dropout `0.1`, and landmark weighting between moves `20` and `50` with factor `1.5`. This model reached about `0.619` test accuracy with test log loss about `0.774`. At move `20`, its landmark accuracy was about `0.642`, and by move `40` it reached about `0.783`. The plain `SimpleRNN(64)` still achieved the highest average landmark accuracy among the earlier untuned recurrent baselines, while the midgame-weighted `GRU(64)` remained strongest on average landmark Brier score. Overall, the sweep showed that recurrent-model tuning changes the tradeoff between calibration, overall test accuracy, and strength around the midgame landmarks rather than producing one universally best architecture.

To make the evaluation more faithful to the proposal's probability-prediction goal, all models were also re-evaluated with landmarking every 5 full moves on the `1,000`-game subsets. This gives one prediction per game at moves `5, 10, 15, ...` and reports landmark accuracy, log loss, and multiclass Brier score. The Brier score is particularly useful because it measures whether predicted probabilities are well judged, not just whether the top class is correct.

The landmark evaluation reinforced the earlier conclusion that pooled snapshot accuracy is not enough on its own. The stronger models should be judged by how they behave over time. The board-aware logistic-regression model achieved average landmark accuracy about `0.607` and average landmark Brier score about `0.492` on `1,000` games, and these improved to about `0.665` and `0.455` respectively on `10,000` games. The calibrated board SVM was close behind at about `0.606` average landmark accuracy on `1,000` games and about `0.657` on `10,000` games, with Brier scores around `0.493` and `0.458`. The basic RNN improved the `1,000`-game landmark averages further to about `0.628` accuracy and `0.471` Brier score, and the `10,000` game RNN improved the overall sequence-based test accuracy to about `0.615` with test log loss about `0.774`. At move `5`, both the logistic and recurrent models remained fairly uncertain, which is appropriate for the opening. Later in the game, both became stronger, with the `10,000` game board logistic baseline, the calibrated SVM, and the recurrent models all crossing roughly the high-`0.60` to low-`0.70` range by moves `30` to `40`. In contrast, the lighter models stayed near `0.44` average landmark accuracy, while several of the MLP and boosted-tree variants often had similar or slightly lower landmark accuracy but much worse log loss and Brier score, indicating poorer probability calibration.

This comparison is useful because it shows that richer features mattered much more than simply switching to a stronger classifier, but it also suggests that sequential structure can add value once the feature set is strong enough. Board-state features remain essential, and the recurrent models provide evidence that temporal modeling may improve on snapshot-only methods when evaluated with landmarks. At the same time, the `10,000`-game board logistic-regression result shows that a simple calibrated snapshot model remains very competitive once the feature engineering and data volume are strong enough. Among the `1,000`-game recurrent variants, the best practical overall choice was the combined `GRU(64)` with dropout and midgame weighting, because it gave the strongest held-out test log loss while also improving the midgame landmarks that matter most for average-length games.

### Overall Comparison

| Model | Dataset | Evaluation | Test Accuracy | Test Log Loss | Avg Landmark Accuracy | Avg Landmark Brier | Move 5 Acc | Move 10 Acc | Move 20 Acc | Move 30 Acc | Move 40 Acc |
|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Light Logistic Regression | 1k | Snapshot | 0.457 | 0.911 | 0.441 | 0.579 | 0.472 | 0.487 | 0.437 | 0.453 | 0.383 |
| Light HistGradientBoosting | 1k | Snapshot | 0.461 | 2.221 | 0.440 | 0.899 | 0.451 | 0.465 | 0.457 | 0.443 | 0.433 |
| Board Logistic Regression | 1k | Snapshot | 0.572 | 0.837 | 0.607 | 0.492 | 0.467 | 0.535 | 0.576 | 0.594 | 0.700 |
| Board Logistic Regression | 10k | Snapshot | 0.611 | 0.795 | 0.665 | 0.455 | 0.525 | 0.559 | 0.639 | 0.694 | 0.683 |
| Board SVM (calibrated linear) | 1k | Snapshot | 0.578 | 0.818 | 0.606 | 0.493 | 0.472 | 0.535 | 0.570 | 0.585 | 0.717 |
| Board SVM (calibrated linear) | 10k | Snapshot | 0.610 | 0.795 | 0.657 | 0.458 | 0.527 | 0.559 | 0.642 | 0.693 | 0.679 |
| Board HistGradientBoosting | 1k | Snapshot | 0.539 | 1.826 | 0.559 | 0.722 | 0.492 | 0.471 | 0.570 | 0.576 | 0.583 |
| Board XGBoost | 10k | Snapshot | 0.598 | 0.844 | N/A | N/A | N/A | N/A | N/A | N/A | N/A |
| Board MLP (128,64) | 1k | Snapshot | 0.541 | 3.280 | 0.556 | 0.772 | 0.487 | 0.481 | 0.517 | 0.519 | 0.583 |
| Board Deep MLP (256,128,64) | 1k | Snapshot | 0.543 | 3.668 | 0.555 | 0.779 | 0.451 | 0.497 | 0.556 | 0.576 | 0.583 |
| Board Deep MLP (256,128,64) | 10k | Snapshot | 0.545 | 3.072 | N/A | N/A | N/A | N/A | N/A | N/A | N/A |
| Board Balanced MLP (256,128,64)+OS | 10k | Snapshot | 0.542 | 2.935 | N/A | N/A | N/A | N/A | N/A | N/A | N/A |
| Board Wide MLP (512,256,128) | 10k | Snapshot | 0.541 | 5.626 | N/A | N/A | N/A | N/A | N/A | N/A | N/A |
| Basic SimpleRNN | 1k | Landmark Sequence | 0.598 | 0.807 | 0.628 | 0.471 | 0.513 | 0.562 | 0.636 | 0.651 | 0.750 |
| Basic SimpleRNN | 10k | Landmark Sequence | 0.615 | 0.774 | 0.672 | 0.424 | 0.529 | 0.536 | 0.628 | 0.687 | 0.701 |
| Basic SimpleRNN | 250k | Landmark Sequence | 0.628 | 0.763 | 0.689 | 0.420 | 0.529 | 0.555 | 0.640 | 0.703 | 0.714 |
| Basic SimpleRNN | 8.63M | Landmark Sequence | 0.636 | 0.749 | 0.742 | 0.353 | 0.534 | 0.563 | 0.650 | 0.709 | 0.724 |
| Best Tuned GRU (dropout + midgame weighting) | 1k | Landmark Sequence | 0.619 | 0.774 | 0.637 | 0.442 | 0.528 | 0.540 | 0.642 | 0.585 | 0.783 |
| Best Tuned GRU (dropout + midgame weighting) | 10k | Landmark Sequence | 0.622 | 0.769 | 0.669 | 0.427 | 0.536 | 0.545 | 0.647 | 0.697 | 0.685 |

### Top 3 Models On 50,000 Games

To test how the strongest current models scale, the board-aware feature extraction pipeline was expanded to `50,000` games, producing `3,165,246` move-level rows. The three models selected for this larger rerun were the best tuned `GRU(64)` variant, the basic `SimpleRNN(64)`, and the board-aware logistic-regression baseline. These were chosen because they were the strongest overall models at the `10,000`-game stage.

| Model | Test Accuracy | Test Log Loss | Avg Landmark Accuracy | Avg Landmark Brier | Move 5 Acc | Move 10 Acc | Move 20 Acc | Move 30 Acc | Move 40 Acc |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Board Logistic Regression (50k) | 0.618 | 0.783 | 0.673 | 0.436 | 0.522 | 0.550 | 0.631 | 0.688 | 0.732 |
| Basic SimpleRNN (50k) | 0.637 | 0.747 | 0.690 | 0.407 | 0.534 | 0.572 | 0.645 | 0.702 | 0.737 |
| Tuned GRU (50k) | 0.610 | 0.809 | 0.596 | 0.536 | 0.523 | 0.539 | 0.618 | 0.672 | 0.717 |

This larger-scale comparison produced a useful surprise. The basic `SimpleRNN` scaled best, improving to about `0.637` test accuracy with test log loss about `0.747`, while the board logistic-regression model also improved to about `0.618` test accuracy and remained a strong calibrated baseline. In contrast, the previously best tuned `GRU` variant did not continue improving at this scale and instead underperformed both the basic recurrent model and the logistic baseline. This suggests that the simpler recurrent setup generalized better on the `50,000`-game experiment, while the more aggressively tuned `GRU` may have been over-specialized to the smaller development setting.

A final scaling test pushed the basic `SimpleRNN` to `100,000` board-aware games using the parallel CPU extractor. The resulting dataset used `71,982` training games, `7,998` validation games, and `19,995` test games. Training was monitored with validation loss each epoch, and the final test metrics were computed from the checkpoint corresponding to the best validation epoch rather than simply from the last epoch. In this rerun, the best validation loss occurred at epoch `3`, after which validation performance flattened slightly while training loss continued to decrease. The selected model reached about `0.631` test accuracy with test log loss about `0.755`, with validation log loss about `0.757`. Landmark accuracy remained strong in the middlegame and later positions, reaching about `0.529` at move `5`, `0.560` at move `10`, `0.644` at move `20`, `0.700` at move `30`, and `0.718` at move `40`.



> **Note for Colab:** The plot images below are generated by the local training scripts and are not tracked in the repository. They will not display in Colab unless regenerated. The underlying data is available in `artifacts/reported_results/`.

*(SimpleRNN 100,000-game loss history — generated locally; not available in Colab)*

A larger follow-up experiment then pushed the basic recurrent model to `250,000` games using the new NPZ shard pipeline and the GPU-enabled PyTorch trainer. This run used `179,701` training games, `20,303` validation games, and `49,937` test games, corresponding to `1,062,775` training landmark prefixes, `119,624` validation prefixes, and `295,391` test prefixes. As with the TensorFlow runs, the PyTorch trainer tracked validation loss each epoch, restored the best-validation model state, and then reported final validation and test metrics from that selected checkpoint. In the rerun with saved history, the best validation loss again occurred at epoch `3`, with validation loss improving quickly in the first few epochs before leveling off. On this setup, the shard-based `SimpleRNN` achieved about `0.628` test accuracy with test log loss about `0.763`, with validation log loss about `0.751`. Its move-level landmark profile remained strong and fairly smooth: about `0.529` at move `5`, `0.555` at move `10`, `0.640` at move `20`, `0.703` at move `30`, and `0.714` at move `40`. This means the `250,000`-game run remained competitive with the earlier best recurrent results, while the loss-history plot also shows that most of the useful learning happened very early in training.

*(SimpleRNN 250,000-game loss history — generated locally; not available in Colab)*

*(Top 3 models on 50k games: landmark accuracy — generated locally; not available in Colab)*

### Basic SimpleRNN Scaling Comparison

Because the basic `SimpleRNN` became the strongest recurrent result at larger scale, it is useful to compare that same architecture directly across dataset sizes. The table below isolates only the basic `SimpleRNN` runs at `1,000`, `10,000`, `50,000`, `100,000`, `250,000`, `1,000,000`, and the final all-games run over `8,633,881` games.

| Dataset | Test Accuracy | Test Log Loss | Avg Landmark Accuracy | Avg Landmark Brier | Move 5 Acc | Move 10 Acc | Move 20 Acc | Move 30 Acc | Move 40 Acc |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1k | 0.598 | 0.807 | 0.628 | 0.471 | 0.492 | 0.471 | 0.623 | 0.604 | 0.783 |
| 10k | 0.615 | 0.774 | 0.672 | 0.424 | 0.529 | 0.536 | 0.627 | 0.687 | 0.701 |
| 50k | 0.637 | 0.747 | 0.690 | 0.407 | 0.534 | 0.572 | 0.645 | 0.702 | 0.737 |
| 100k | 0.631 | 0.755 | 0.700 | 0.403 | 0.529 | 0.560 | 0.644 | 0.700 | 0.718 |
| 250k | 0.628 | 0.763 | 0.689 | 0.420 | 0.529 | 0.555 | 0.640 | 0.703 | 0.714 |
| 1M | 0.634 | 0.753 | 0.704 | 0.396 | 0.532 | 0.562 | 0.646 | 0.706 | 0.722 |
| 8.63M | 0.636 | 0.749 | 0.742 | 0.353 | 0.534 | 0.563 | 0.650 | 0.709 | 0.724 |

This focused view makes the scaling pattern easier to interpret. The basic `SimpleRNN` improved clearly from `1k` to `10k`, improved again at `50k`, and then remained strong through `100k`, `250k`, `1M`, and the final all-games run. The full `8.63M` run now delivers the strongest overall held-out test accuracy, the best average landmark accuracy, and the best average landmark Brier score among the SimpleRNN scaling experiments. That suggests the model continues to benefit from scale not only in stage-wise quality, but also in its final pooled held-out performance once the full dataset is used.

*(Basic SimpleRNN scaling comparison — generated locally; not available in Colab)*

### RNN Variant Sweep On 1,000 Games
> **Note on the sweep results:** The Test Accuracy and Test Log Loss shown in this sweep table (`0.614` / `0.815` for the plain SimpleRNN) differ slightly from the `1k` row in the Overall Comparison table (`0.598` / `0.807`). Both are from the same `1,000`-game board-aware dataset but from separate training runs with different random seeds. The sweep is reported as-is so the relative comparison between variants remains valid.


| RNN Variant | Key Settings | Test Accuracy | Test Log Loss | Avg Landmark Accuracy | Avg Landmark Brier | Move 5 Acc | Move 10 Acc | Move 20 Acc | Move 30 Acc | Move 40 Acc |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| SimpleRNN | `SimpleRNN(64)`, `lr=1e-3`, `batch=64` | 0.614 | 0.815 | 0.638 | 0.478 | 0.513 | 0.562 | 0.636 | 0.651 | 0.750 |
| GRU | `GRU(64)`, `lr=1e-3`, `batch=64` | 0.582 | 0.842 | 0.603 | 0.486 | N/A | N/A | N/A | N/A | N/A |
| LSTM | `LSTM(64)`, `lr=1e-3`, `batch=64` | 0.585 | 0.821 | 0.616 | 0.488 | N/A | N/A | N/A | N/A | N/A |
| Stacked GRU | `GRU(64,64)`, `lr=1e-3`, `batch=64` | 0.595 | 0.820 | 0.624 | 0.476 | N/A | N/A | N/A | N/A | N/A |
| GRU + Dropout | `GRU(64)`, `dropout=0.2`, `rec_dropout=0.1` | 0.613 | 0.787 | 0.633 | 0.468 | N/A | N/A | N/A | N/A | N/A |
| GRU Tuned LR | `GRU(64)`, `lr=5e-4`, `batch=128`, `epochs=40` | 0.586 | 0.832 | 0.610 | 0.490 | N/A | N/A | N/A | N/A | N/A |
| GRU Midgame Weighted | `GRU(64)`, weight moves `20-50` by `1.5` | 0.607 | 0.790 | 0.634 | 0.458 | N/A | N/A | N/A | N/A | N/A |
| GRU Draw Weighted | `GRU(64)`, draw weight `2.0` | 0.578 | 0.838 | 0.593 | 0.493 | N/A | N/A | N/A | N/A | N/A |
| GRU Dropout + Midgame Weighting | `GRU(64)`, `dropout=0.2`, `rec_dropout=0.1`, weight moves `20-50` by `1.5` | 0.619 | 0.774 | 0.637 | 0.442 | 0.528 | 0.540 | 0.642 | 0.585 | 0.783 |

### Landmark Accuracy Curves

The following figure overlays landmark accuracy at moves `5, 10, 15, ...` for every saved model that had landmark-level evaluation available. This makes it easier to compare how quickly each model becomes informative as the game develops, and it visually reinforces that the strongest recurrent and board-aware models separate themselves mostly after the opening.

*(Landmark accuracy comparison across all saved models — generated locally; not available in Colab)*

A natural next step is to add richer chess-state information, such as board representations, material balance, or legal-move context, and then compare whether those additions improve predictive performance beyond the current Elo-and-clock baseline.


## **Conclusions**

The experiments show that human-centered win prediction benefits substantially from combining practical features such as Elo and clock usage with explicit board-state information. Across the development experiments, the strongest snapshot result came from board-aware logistic regression, while the calibrated board SVM emerged as a competitive additional tabular baseline and the recurrent models became strongest on the sequence-based landmark evaluation.

A key conclusion is that feature quality mattered more than model complexity in the current stage of the project. Adding real chess-state features produced a large improvement, while simply switching to more flexible models did not automatically help. The current results therefore support a project direction focused on stronger chess representations, careful probability calibration, and evaluation across different game stages.

The landmarking results are especially helpful for interpretation. They show that the model is not simply producing one pooled accuracy number over the whole game. Instead, prediction quality is low in the opening, where uncertainty is natural, and rises steadily later in the game, which is exactly the qualitative behavior expected from a realistic win-probability model.

At the same time, the draw class remains difficult, and not all neural models behaved well. The MLP-style networks generally underperformed the best snapshot baseline in calibration, even when scaled to more data. The calibrated SVM was much more competitive than the MLP family and came very close to the `10,000`-game board logistic baseline, but it still did not clearly beat it. However, the recurrent models provided a more encouraging result by exploiting sequence information directly and outperforming the board-aware logistic model under the landmark-based evaluation. Scaling the RNN from `1,000` to `10,000` games improved the sequence-based test accuracy from about `0.598` to about `0.615`, and the `50,000`-game SimpleRNN improved further to about `0.637`. The later `100,000`-game, shard-based `250,000`-game, `1,000,000`-game, and final all-games `8.63M` SimpleRNN runs remained strong at about `0.631`, `0.628`, `0.634`, and `0.636` respectively. The final all-games run produced the strongest held-out accuracy, the strongest average landmark accuracy, and the strongest average Brier score so far among the SimpleRNN scaling experiments. This suggests that the current simple recurrent setup continues to benefit from scale, although the improvements become more gradual at the largest sizes.

A practical limitation is full-scale training on all `8.63M` filtered games with board-aware reconstruction. A benchmark of a streaming out-of-core logistic-style trainer on the full filtered PGN processed about `2,000` games and `12,017` sampled landmark positions in about `105` seconds, or roughly `115` sampled positions per second. At that rate, a true all-games board-aware pass would take on the order of several days in this environment rather than a couple of hours. Parallel CPU extraction helped substantially, reducing a `1,000`-game board-aware extraction benchmark from about `56.4` seconds to about `19.8` seconds with `4` workers. Later, a shard-based NPZ sequence pipeline plus a CUDA-enabled PyTorch SimpleRNN trainer made it possible to scale the recurrent experiments all the way to the full filtered dataset. This produced a complete all-games run over `8.63M` rapid games with no-leakage splits and landmark evaluation. Even so, the full board-aware pipeline remains computationally heavy, which is why the report still documents the intermediate subset experiments as important stages of the scaling story.

This tradeoff also clarifies the realistic next steps. If the goal is to use all available games, the most practical option is an out-of-core logistic-style model with lighter non-board features, or a more aggressive landmark sampling policy, rather than full board reconstruction on every game. If the goal is to preserve the strongest feature quality, the current results now show that the sharded GPU-based recurrent pipeline can be pushed well beyond the earlier subset limits, up to the full filtered dataset, although at substantial computational cost. This means the project is not finished, but the current pipeline is already strong enough to support a clear, defensible experimental story: richer board-aware features improve prediction quality, sequence-aware models may improve further when evaluated at realistic landmarks, and careful no-leakage evaluation is essential for interpreting the results correctly.


## **References**
- Lichess Open Database. Official website and data source: https://database.lichess.org/ . Used as the raw PGN source for all experiments.
- `python-chess` documentation: https://python-chess.readthedocs.io/ . Used for PGN parsing and legal board reconstruction.
- scikit-learn documentation: https://scikit-learn.org/stable/ . Used for logistic regression, calibrated SVM, histogram gradient boosting, and MLP baselines.
- TensorFlow documentation: https://www.tensorflow.org/ and Keras documentation: https://keras.io/ . Used for the earlier recurrent-neural-network experiments.
- PyTorch documentation: https://pytorch.org/docs/stable/index.html . Used for the shard-based GPU SimpleRNN experiments at larger scale.
- XGBoost documentation: https://xgboost.readthedocs.io/ . Used for the later board-aware boosted-tree comparison.
- Course Project Guidelines document and Course Material
